# Notebook 05g — RF Gallery for m=1 Simple Cells

Visualisation requested by Prof. Lindeberg: for all 31 verified m=1 neurons,
side-by-side raw RF map and idealized Gaussian derivative model with fitted parameters.

**Paths and merge logic are identical to nb05d.**

In [1]:
import sys, warnings, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.linear_model import RidgeCV, Ridge
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

sys.path.insert(0, os.path.expanduser('~/dev/neuroscience/v1-dimensionality-study/src'))
from rf_analysis.sparse_noise import (
    fit_rf_by_order,
    gaussian_deriv_m1,
    estimate_phi_from_lobes,
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Imports OK')

AttributeError: `np.unicode_` was removed in the NumPy 2.0 release. Use `np.str_` instead.

In [ ]:
# ---------------------------------------------------------------------------
# Paths — identical to nb05d Cell 2
# ---------------------------------------------------------------------------
base_dir    = Path(os.path.expanduser('~/dev/neuroscience/v1-dimensionality-study'))
cache_dir   = base_dir / 'data' / 'cache'
lists_dir   = base_dir / 'data' / 'experiment_lists'
outputs_dir = base_dir / 'outputs' / 'rf_params'

nb05d_dir = outputs_dir / 'order_full_v2'   # v2 rerun CSVs + judgements
nb07_dir  = outputs_dir / 'order_full'       # nb07 manual judgements
nb09_dir  = outputs_dir / 'nb09'             # nb09 manual judgements

gallery_dir = nb05d_dir / 'rf_gallery'
gallery_dir.mkdir(parents=True, exist_ok=True)

manifest_path = str(cache_dir / 'manifest.json')
boc = BrainObservatoryCache(manifest_file=manifest_path)

batch_df       = pd.read_csv(lists_dir / 'all_l23_excitatory_experiments.csv')
verify_df      = pd.read_csv(lists_dir / 'downloaded_all_l23_excitatory_experiments.csv')
downloaded     = set(verify_df[verify_df['exists']]['experiment_id'].tolist())
container_meta = pd.read_csv(lists_dir / 'container_neuron_counts.csv')
complete_ids   = set(container_meta['container_id'].tolist())

session_c = batch_df[
    batch_df['session_type'].str.contains('C') &
    batch_df['id'].isin(downloaded) &
    batch_df['experiment_container_id'].isin(complete_ids)
].copy()

container_to_exp  = dict(zip(session_c['experiment_container_id'], session_c['id']))
container_meta_d  = dict(zip(session_c['experiment_container_id'],
                              zip(session_c['cre_line'], session_c['imaging_depth'])))

print(f'Session C experiments: {len(session_c)}')
print(f'Containers: {len(container_to_exp)}')

In [ ]:
# ---------------------------------------------------------------------------
# batch_get_rf_maps_safe — exact copy from nb05d Cell 3
# ---------------------------------------------------------------------------
def batch_get_rf_maps_safe(dataset, requested_ids,
                            n_lags=8, response_delay=4, response_window=5):
    from rf_analysis.sparse_noise import _build_design_matrix, _pixel_size, _LAMBDA_GRID
    exp_cells = list(dataset.get_cell_specimen_ids())
    req_set   = set(int(c) for c in requested_ids)
    valid_ids = [c for c in exp_cells if c in req_set]
    if not valid_ids:
        raise ValueError('No requested cell_ids found in experiment.')

    stim_name = next((s for s in dataset.list_stimuli() if 'sparse_noise' in s), None)
    if stim_name is None:
        raise ValueError('No sparse noise stimulus.')

    stim_table = dataset.get_stimulus_table(stim_name)
    tr         = dataset.get_locally_sparse_noise_stimulus_template(stimulus=stim_name)
    template   = tr[0] if isinstance(tr, tuple) else tr
    n_tf, grid_h, grid_w = template.shape
    on_val, off_val = int(template.max()), int(template.min())
    pix_size        = _pixel_size(stim_name)

    starts        = stim_table['start'].values.astype(int)
    frame_indices = stim_table['frame'].values.astype(int)

    _, all_dff    = dataset.get_dff_traces(cell_specimen_ids=valid_ids)
    n_cells, n_tp = all_dff.shape

    X = _build_design_matrix(template, frame_indices, on_val, off_val,
                              grid_h, grid_w, n_lags)

    win_idx  = (starts + response_delay)[:, None] + np.arange(response_window)
    in_bnds  = (win_idx >= 0) & (win_idx < n_tp)
    win_safe = np.clip(win_idx, 0, n_tp - 1)
    dff_wins = all_dff[:, win_safe]
    dff_wins[:, ~in_bnds] = np.nan
    Y = np.nanmean(dff_wins, axis=2).T
    Y = np.where(np.isnan(Y), 0.0, Y).astype(np.float64)

    frame_valid  = (frame_indices >= 0) & (frame_indices < n_tf)
    X_fit, Y_fit = X[frame_valid].astype(np.float64), Y[frame_valid]
    if X_fit.shape[0] < 20:
        raise ValueError('Too few valid presentations.')

    try:
        rcv = RidgeCV(alphas=_LAMBDA_GRID, cv=None,
                      fit_intercept=True, alpha_per_target=True)
        rcv.fit(X_fit, Y_fit)
        W = np.atleast_2d(rcv.coef_)
    except TypeError:
        rcv = RidgeCV(alphas=_LAMBDA_GRID, cv=None, fit_intercept=True)
        rcv.fit(X_fit, Y_fit.mean(axis=1, keepdims=True))
        r = Ridge(alpha=float(rcv.alpha_), fit_intercept=True)
        r.fit(X_fit, Y_fit)
        W = np.atleast_2d(r.coef_)

    rf_out = {}
    for i, cid in enumerate(valid_ids):
        strf     = W[i].reshape(n_lags, grid_h, grid_w)
        best_lag = int(np.argmax([np.abs(strf[l]).max() for l in range(n_lags)]))
        rf_out[cid] = (strf[best_lag], pix_size)
    return rf_out

print('batch_get_rf_maps_safe defined')

In [ ]:
# ---------------------------------------------------------------------------
# Load v2 population CSV and rebuild m_final
# Exact copy of nb05d Cell 5 + Cell 6 merge logic
# ---------------------------------------------------------------------------
csv_files = sorted(nb05d_dir.glob('rf_params_order_v2_container_*.csv'))
print(f'Found {len(csv_files)} container CSVs')

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    if 'container_id' not in df.columns:
        df['container_id'] = int(f.stem.split('_')[-1])
    dfs.append(df)
all_df = pd.concat(dfs, ignore_index=True)

for col in ['sigma', 'theta', 'kappa', 'r_squared', 'sigma_x', 'sigma_y',
            'x0', 'y0', 'phi', 'phi_confidence', 'theta_hybrid',
            'cortex_x_um', 'cortex_y_um', 'r2_m0', 'r2_m1', 'r2_m2',
            'delta_r2_vs_m0']:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors='coerce')

all_df['sigma_major'] = all_df[['sigma_x', 'sigma_y']].max(axis=1)
all_df['sigma_minor'] = all_df[['sigma_x', 'sigma_y']].min(axis=1)
all_df['log_sigma']   = np.log10(all_df['sigma'])
all_df['log_kappa']   = np.log(all_df['kappa'])

# theta_hybrid for m=0 and m=2 falls back to theta
all_df.loc[all_df['derivative_order'] == 0, 'theta_hybrid'] = \
    all_df.loc[all_df['derivative_order'] == 0, 'theta']
all_df.loc[all_df['derivative_order'] == 2, 'theta_hybrid'] = \
    all_df.loc[all_df['derivative_order'] == 2, 'theta']

print(f'Total neurons: {len(all_df):,}  |  {all_df["container_id"].nunique()} containers')

In [ ]:
# ---------------------------------------------------------------------------
# m_final merge — exact copy of nb05d Cell 6
# nb07 m=1 calls are protected: later reviews cannot demote them
# ---------------------------------------------------------------------------

# Pass 1: nb07 as protected base
judged      = {}
nb07_m1_ids = set()
nb07_path   = nb07_dir / 'manual_m_judgements.csv'
if nb07_path.exists():
    jdf = pd.read_csv(nb07_path)
    jdf = jdf[jdf['m_manual'] >= 0]
    for _, r in jdf.iterrows():
        judged[int(r['cell_id'])] = int(r['m_manual'])
        if int(r['m_manual']) == 1:
            nb07_m1_ids.add(int(r['cell_id']))
    print(f'nb07: {len(jdf)} judgements  ({len(nb07_m1_ids)} m=1 protected)')
else:
    print('nb07: not found')

# Pass 2: later review files — cannot demote nb07 m=1
later_sources = [
    (nb09_dir  / 'nb09_review_judgements.csv',       'nb09',          'm_manual'),
    (nb05d_dir / 'targeted_review_judgements.csv',   'targeted',      'm_manual'),
    (nb05d_dir / 'unreviewed_m1_judgements.csv',     'unreviewed_m1', 'm_manual'),
    (nb05d_dir / 'rescue_review_judgements.csv',     'rescue',        'm_final'),
]
for path, label, col in later_sources:
    if not path.exists():
        print(f'  {label}: not found — skipping')
        continue
    jdf     = pd.read_csv(path)
    jdf     = jdf[jdf[col] >= 0]
    applied = skipped = 0
    for _, r in jdf.iterrows():
        cid   = int(r['cell_id'])
        m_new = int(r[col])
        if cid in nb07_m1_ids and m_new == 0:
            skipped += 1
            continue
        judged[cid] = m_new
        applied += 1
    print(f'{label}: {applied} applied, {skipped} skipped (protected nb07 m=1)')

all_df['m_manual']          = all_df['cell_id'].map(judged)
all_df['manually_verified'] = all_df['cell_id'].isin(judged)
all_df['m_final'] = np.where(
    all_df['m_manual'].notna(),
    all_df['m_manual'],
    all_df['derivative_order']
).astype(int)

# Auto-demote unreviewed m=2 with marginal ΔR²
if 'delta_r2_vs_m0' in all_df.columns:
    demote = (
        (all_df['m_final'] == 2) &
        (~all_df['manually_verified']) &
        (all_df['delta_r2_vs_m0'] < 0.10)
    )
    all_df.loc[demote, 'm_final'] = 0
    print(f'Auto-demoted {demote.sum()} unreviewed m=2 → m=0')

m1 = all_df[all_df['m_final'] == 1].copy()
print(f'\nm=1: {len(m1)}  (target: 31)')

In [ ]:
# ---------------------------------------------------------------------------
# Load RF maps for all m=1 neurons
# ---------------------------------------------------------------------------
rf_map_cache  = {}
dataset_cache = {}

by_container = m1.groupby('container_id')['cell_id'].apply(list).to_dict()
print(f'Loading {len(m1)} neurons across {len(by_container)} containers')

t_start = time.time()
for i, (container_id, cids) in enumerate(by_container.items()):
    if container_id not in container_to_exp:
        print(f'  [{i+1}/{len(by_container)}] {container_id}: not in session_c — skip')
        continue
    print(f'  [{i+1}/{len(by_container)}] Container {container_id} '
          f'({len(cids)} neurons) ...', end=' ', flush=True)
    try:
        if container_id not in dataset_cache:
            dataset_cache[container_id] = boc.get_ophys_experiment_data(
                container_to_exp[container_id])
        maps = batch_get_rf_maps_safe(dataset_cache[container_id], cids)
        rf_map_cache.update(maps)
        print(f'{len(maps)} loaded  ({time.time()-t_start:.0f}s)')
    except Exception as e:
        print(f'FAIL — {e}')

print(f'\nTotal RF maps loaded: {len(rf_map_cache)}/{len(m1)}')

In [ ]:
# ---------------------------------------------------------------------------
# Refit m=1 Gaussian derivative for each neuron
# ---------------------------------------------------------------------------
N_RANDOM_STARTS = 15
SMOOTH_SIGMA    = 0.75

fit_results = {}   # cell_id → (params, quality, fitted_map, pix)

for idx, row in m1.reset_index(drop=True).iterrows():
    cid = int(row['cell_id'])
    if cid not in rf_map_cache:
        print(f'  [{idx+1}/{len(m1)}] cell {cid}: no RF map — skip')
        continue
    rf_map, pix = rf_map_cache[cid]
    rf_on  = np.maximum(rf_map,  0.0)
    rf_off = np.maximum(-rf_map, 0.0)
    grid_h, grid_w = rf_map.shape
    Y_g, X_g = np.mgrid[0:grid_h, 0:grid_w].astype(float)
    xy = (X_g, Y_g)
    try:
        p, q = fit_rf_by_order(
            rf_on, rf_off, m=1,
            pixel_size_deg=pix,
            smooth_sigma=SMOOTH_SIGMA,
            n_random_starts=N_RANDOM_STARTS,
            seed=cid,
        )
        su_px = p['sigma_x'] / pix
        sv_px = p['sigma_y'] / pix
        x0_px = p['x0']     / pix
        y0_px = p['y0']     / pix
        th_r  = np.deg2rad(p['theta'])
        popt  = [p['amplitude'], x0_px, y0_px, su_px, sv_px, th_r, 0.0]
        fitted = gaussian_deriv_m1(xy, *popt).reshape(grid_h, grid_w)
        phi_fresh, _ = estimate_phi_from_lobes(
            rf_on, rf_off, m=1,
            pixel_size_deg=pix,
            smooth_sigma=SMOOTH_SIGMA,
        )
        fit_results[cid] = (p, q, fitted, pix, phi_fresh)
        print(f'  [{idx+1}/{len(m1)}] cell {cid}:  '
              f'σ={p["sigma"]:.1f}°  κ={p["kappa"]:.2f}  '
              f'θ={row["theta_hybrid"]:.0f}°  R²={q["r_squared"]:.3f}')
    except Exception as e:
        print(f'  [{idx+1}/{len(m1)}] cell {cid}: FAIL — {e}')

print(f'\nFit complete: {len(fit_results)}/{len(m1)} neurons')

In [ ]:
# ---------------------------------------------------------------------------
# Fig 1: RF gallery sorted by κ ascending (near-circular → most elongated)
# As requested by Prof. Lindeberg:
# Left column: raw ridge RF map (input to model fitting)
# Right column: idealized Gaussian derivative model (m=1)
# Parameters shown: σ, κ, θ (φ lobe-geometry estimator), R²
# ---------------------------------------------------------------------------
def draw_orientation_bar(ax, phi_deg, cx, cy, half_len, color='white', lw=1.5):
    # phi_deg is the differentiation axis (φ, lobe-geometry estimator).
    # Edge orientation = φ + 90° — the bar shows the preferred EDGE, not
    # the differentiation direction.
    edge_deg = (phi_deg + 90) % 180
    th = np.deg2rad(edge_deg)
    dx, dy = np.cos(th) * half_len, np.sin(th) * half_len
    ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy],
            color=color, lw=lw, solid_capstyle='round')

plot_df = (
    m1[m1['cell_id'].isin(fit_results)]
    .sort_values('kappa', ascending=True)
    .reset_index(drop=True)
)
n_neurons = len(plot_df)
N_COLS    = 4
n_rows    = int(np.ceil(n_neurons / N_COLS))

fig = plt.figure(figsize=(N_COLS * 4.2, n_rows * 2.1), facecolor='white')
outer = gridspec.GridSpec(n_rows, N_COLS, figure=fig, hspace=0.28, wspace=0.10)

for idx, row in plot_df.iterrows():
    cid = int(row['cell_id'])
    p, q, fitted, pix, phi_fresh = fit_results[cid]
    rf_map = rf_map_cache[cid][0]

    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[idx // N_COLS, idx % N_COLS], wspace=0.04)

    vmax = np.nanpercentile(np.abs(rf_map), 98)
    imkw = dict(cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                aspect='equal', origin='lower', interpolation='nearest')

    # Use phi from the fresh refit (lobe-geometry estimator for m=1).
    # Fall back to theta_hybrid from CSV only if phi_fresh is unavailable.
    phi_disp = phi_fresh if (phi_fresh is not None and not np.isnan(float(phi_fresh))) \
               else (float(row['theta_hybrid']) if pd.notna(row['theta_hybrid']) else float(p['theta']))
    r2_col = '#1a7a4a' if q['r_squared'] >= 0.5 else '#888888'
    cre_short = str(row.get('cre_line', '')).replace(
        'Cux2-CreERT2', 'Cux2').replace('Slc17a7-IRES2-Cre', 'Slc17a7')

    # Raw RF
    ax_r = fig.add_subplot(inner[0, 0])
    ax_r.imshow(rf_map, **imkw)
    ax_r.set_xticks([]); ax_r.set_yticks([])
    for sp in ax_r.spines.values():
        sp.set_edgecolor('#bbbbbb'); sp.set_linewidth(0.5)
    ax_r.set_title(f'#{idx+1}  {cre_short}', fontsize=6.5,
                   color='#444', pad=2, loc='left')
    ax_r.text(0.04, 0.96, 'raw RF', transform=ax_r.transAxes,
              fontsize=5.5, color='white', va='top',
              bbox=dict(facecolor='#333', alpha=0.65, pad=1, edgecolor='none'))

    # Fitted model
    ax_f = fig.add_subplot(inner[0, 1])
    ax_f.imshow(fitted, **imkw)
    ax_f.set_xticks([]); ax_f.set_yticks([])
    for sp in ax_f.spines.values():
        sp.set_edgecolor('#2196F3'); sp.set_linewidth(1.0)

    # Orientation bar through RF centre
    cx_px = p['x0'] / pix
    cy_px = p['y0'] / pix
    draw_orientation_bar(ax_f, phi_disp, cx_px, cy_px,
                         max(rf_map.shape) * 0.22)
    ax_f.text(0.04, 0.96, 'model', transform=ax_f.transAxes,
              fontsize=5.5, color='white', va='top',
              bbox=dict(facecolor='#1565C0', alpha=0.7, pad=1, edgecolor='none'))
    ax_f.set_title(
        f'σ={p["sigma"]:.1f}°  κ={p["kappa"]:.2f}\n'
        f'φ={phi_disp:.0f}°  R²={q["r_squared"]:.2f}',
        fontsize=6.2, color=r2_col, pad=2
    )

fig.suptitle(
    f'Mouse V1 m=1 edge-detector simple cells  (n={n_neurons}, all manually verified)\n'
    'Left: raw ridge RF map (input to fitting)   |   '
    'Right: Gaussian derivative model (m=1, Lindeberg framework)\n'
    'σ = spatial scale (°),  κ = elongation,  '
    'θ = orientation (φ lobe-geometry estimator),  R² = fit quality\n'
    'White bar = preferred edge orientation (φ+90°).  Sorted by κ ascending '
    '(near-circular left → most elongated right).',
    fontsize=10, y=1.02, color='#1a1a2e'
)

out = gallery_dir / 'fig_rf_gallery_m1_by_kappa.png'
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.savefig(out.with_name('fig_rf_gallery_m1_by_kappa_hires.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {gallery_dir}')

In [ ]:
# ---------------------------------------------------------------------------
# Fig 2: RF gallery sorted by θ_hybrid ascending
# Shows the uniform spread of preferred orientations across the population
# ---------------------------------------------------------------------------
plot_df2 = (
    m1[m1['cell_id'].isin(fit_results)]
    .sort_values('theta_hybrid', ascending=True)
    .reset_index(drop=True)
)

fig2 = plt.figure(figsize=(N_COLS * 4.2, n_rows * 2.1), facecolor='white')
outer2 = gridspec.GridSpec(n_rows, N_COLS, figure=fig2, hspace=0.28, wspace=0.10)

for idx, row in plot_df2.iterrows():
    cid = int(row['cell_id'])
    p, q, fitted, pix, phi_fresh = fit_results[cid]
    rf_map = rf_map_cache[cid][0]

    inner2 = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer2[idx // N_COLS, idx % N_COLS], wspace=0.04)

    vmax = np.nanpercentile(np.abs(rf_map), 98)
    imkw = dict(cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                aspect='equal', origin='lower', interpolation='nearest')

    phi_disp = phi_fresh if (phi_fresh is not None and not np.isnan(float(phi_fresh))) \
               else (float(row['theta_hybrid']) if pd.notna(row['theta_hybrid']) else float(p['theta']))
    r2_col = '#1a7a4a' if q['r_squared'] >= 0.5 else '#888888'

    ax_r = fig2.add_subplot(inner2[0, 0])
    ax_r.imshow(rf_map, **imkw)
    ax_r.set_xticks([]); ax_r.set_yticks([])
    for sp in ax_r.spines.values():
        sp.set_edgecolor('#bbbbbb'); sp.set_linewidth(0.5)
    ax_r.set_title(f'#{idx+1}', fontsize=6.5, color='#444', pad=2, loc='left')
    ax_r.text(0.04, 0.96, 'raw RF', transform=ax_r.transAxes,
              fontsize=5.5, color='white', va='top',
              bbox=dict(facecolor='#333', alpha=0.65, pad=1, edgecolor='none'))

    ax_f = fig2.add_subplot(inner2[0, 1])
    ax_f.imshow(fitted, **imkw)
    ax_f.set_xticks([]); ax_f.set_yticks([])
    for sp in ax_f.spines.values():
        sp.set_edgecolor('#2196F3'); sp.set_linewidth(1.0)
    draw_orientation_bar(ax_f, phi_disp,
                         p['x0'] / pix, p['y0'] / pix,
                         max(rf_map.shape) * 0.22)
    ax_f.text(0.04, 0.96, 'model', transform=ax_f.transAxes,
              fontsize=5.5, color='white', va='top',
              bbox=dict(facecolor='#1565C0', alpha=0.7, pad=1, edgecolor='none'))
    ax_f.set_title(
        f'σ={p["sigma"]:.1f}°  κ={p["kappa"]:.2f}\n'
        f'φ={phi_disp:.0f}°  R²={q["r_squared"]:.2f}',
        fontsize=6.2, color=r2_col, pad=2
    )

fig2.suptitle(
    f'Mouse V1 m=1 edge-detector simple cells  (n={n_neurons}, all manually verified)\n'
    'Left: raw ridge RF map (input to fitting)   |   '
    'Right: Gaussian derivative model (m=1, Lindeberg framework)\n'
    'σ = spatial scale (°),  κ = elongation,  '
    'θ = orientation (φ lobe-geometry estimator),  R² = fit quality\n'
    'White bar = preferred edge orientation (φ+90°).  Sorted by φ ascending '
    '(0° horizontal → 180°), illustrating uniform orientation coverage.',
    fontsize=9, y=1.02, color='#1a1a2e'
)

out2 = gallery_dir / 'fig_rf_gallery_m1_by_orientation.png'
plt.savefig(out2, dpi=180, bbox_inches='tight', facecolor='white')
plt.savefig(out2.with_name('fig_rf_gallery_m1_by_orientation_hires.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {gallery_dir}')